In [ ]:
pip install torch numpy scikit-learn tqdm

In [32]:
import os
import csv
import pandas as pd
from collections import Counter

print("📦 Loading Test Data...")
print("=" * 60)

TEST_FILE = '/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_test.txt'
SAMPLE_SUB = '/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv'

def load_raw_text(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    data = []
    for i, char in enumerate(content):
        is_ws = (char == ' ' or char == '\n' or char == '\t')
        data.append({'id': i+1, 'char': char, 'is_whitespace': is_ws})
    
    print(f"   Total characters: {len(data)}")
    print(f"   Non-whitespace: {sum(1 for d in data if not d['is_whitespace'])}")
    return data, content

test_data, test_content = load_raw_text(TEST_FILE)

# โหลด sample submission เพื่อตรวจสอบ format
sample_df = pd.read_csv(SAMPLE_SUB)
print(f"\n📋 Sample submission: {len(sample_df)} rows")

📦 Loading Test Data...
   Total characters: 37248
   Non-whitespace: 35182

📋 Sample submission: 35182 rows


In [33]:
# กฎภาษาไทยสำหรับปรับปรุงคำทำนาย

# พยัญชนะไทย 44 ตัว
THAI_CONSONANTS = set('กขฃคฅฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ')

# สระไทย
THAI_VOWELS = set('ะัาำิีึืุูเแโใไๅ็่้๊๋์ฯ')

# สระที่อยู่หน้าพยัญชนะ
THAI_LEADING_VOWELS = set('เแโใไ')

# ตัวการันต์
THAI_THANTHAKHAT = '์'

# วรรณยุกต์
THAI_TONES = set('่้๊๋')

# อักษรสูง กลาง ต่ำ (สำหรับวิเคราะห์)
THAI_HIGH_CONSONANTS = set('ขฃฉฐผฝศษสหฬ')
THAI_MID_CONSONANTS = set('กจดตบปอฎฏ')
THAI_LOW_CONSONANTS = set('คฅฆงชซฌญฑฒณทธนพฟภมยรลวฮ')

print("✅ Thai language rules loaded!")
print(f"   Consonants: {len(THAI_CONSONANTS)}")
print(f"   Vowels: {len(THAI_VOWELS)}")

✅ Thai language rules loaded!
   Consonants: 44
   Vowels: 23


In [34]:
# โหลด predictions จากโมเดลที่คุณฝึกไว้ (0.87918)
# ถ้ามีไฟล์ submission อยู่แล้ว

import pandas as pd

print("\n📥 Loading Your Current Predictions...")
print("=" * 60)

# โหลด submission เดิมของคุณ
YOUR_SUBMISSION = 'submission.csv'  # เปลี่ยนเป็นชื่อไฟล์ของคุณ

if os.path.exists(YOUR_SUBMISSION):
    df = pd.read_csv(YOUR_SUBMISSION)
    predictions = dict(zip(df['Id'], df['Predicted']))
    print(f"   Loaded {len(predictions)} predictions")
    print(f"   Tag distribution: {dict(Counter(predictions.values()))}")
else:
    print("⚠️ No existing submission found!")
    print("   Creating baseline predictions...")
    
    # สร้าง baseline predictions
    predictions = {}
    non_ws = [d for d in test_data if not d['is_whitespace']]
    
    # Split by whitespace (baseline)
    words = []
    current_word = []
    for item in test_data:
        if item['is_whitespace']:
            if current_word:
                words.append(current_word)
                current_word = []
        else:
            current_word.append(item)
    if current_word:
        words.append(current_word)
    
    for word in words:
        if len(word) == 1:
            predictions[word[0]['id']] = 'E_WORD'
        elif len(word) == 2:
            predictions[word[0]['id']] = 'B_WORD'
            predictions[word[1]['id']] = 'E_WORD'
        else:
            predictions[word[0]['id']] = 'B_WORD'
            for k in range(1, len(word) - 1):
                predictions[word[k]['id']] = 'I_WORD'
            predictions[word[-1]['id']] = 'E_WORD'
    
    print(f"   Created {len(predictions)} baseline predictions")


📥 Loading Your Current Predictions...
   Loaded 35182 predictions
   Tag distribution: {'B_WORD': 7095, 'I_WORD': 20687, 'E_WORD': 7400}


In [36]:
def apply_thai_post_processing(predictions, test_data):
    """
    ใช้กฎภาษาไทยแก้ไข predictions
    คาดว่าจะเพิ่ม F1 +0.02-0.05
    """
    print("\n🔧 Applying Thai Post-Processing Rules...")
    print("=" * 60)
    
    # สร้าง id → char mapping
    id_to_char = {d['id']: d['char'] for d in test_data}
    id_to_ws = {d['id']: d['is_whitespace'] for d in test_data}
    
    # Group predictions by sequences (ระหว่าง whitespace)
    sequences = []
    current_seq = []
    
    for item in sorted(test_data, key=lambda x: x['id']):
        if item['is_whitespace']:
            if current_seq:
                sequences.append(current_seq)
            current_seq = []
        else:
            current_seq.append({
                'id': item['id'],
                'char': item['char'],
                'tag': predictions.get(item['id'], 'E_WORD')
            })
    
    if current_seq:
        sequences.append(current_seq)
    
    print(f"   Found {len(sequences)} word sequences")
    
    # Apply rules
    corrections = 0
    
    for seq_idx, seq in enumerate(sequences):
        if len(seq) == 0:
            continue
        
        original_tags = [s['tag'] for s in seq]
        
        # Rule 1: คำ 1 ตัวอักษร → E_WORD
        if len(seq) == 1:
            if seq[0]['tag'] != 'E_WORD':
                seq[0]['tag'] = 'E_WORD'
                corrections += 1
        
        # Rule 2: คำ 2 ตัวอักษร → B_WORD + E_WORD
        elif len(seq) == 2:
            if seq[0]['tag'] != 'B_WORD':
                seq[0]['tag'] = 'B_WORD'
                corrections += 1
            if seq[1]['tag'] != 'E_WORD':
                seq[1]['tag'] = 'E_WORD'
                corrections += 1
        
        # Rule 3: คำ 3+ ตัวอักษร → B_WORD + I_WORD* + E_WORD
        elif len(seq) >= 3:
            # ตรวจสอบว่าเริ่มด้วยพยัญชนะหรือไม่
            if seq[0]['char'] in THAI_CONSONANTS:
                if seq[0]['tag'] != 'B_WORD':
                    seq[0]['tag'] = 'B_WORD'
                    corrections += 1
            
            # ตัวสุดท้ายต้องเป็น E_WORD
            if seq[-1]['tag'] != 'E_WORD':
                seq[-1]['tag'] = 'E_WORD'
                corrections += 1
            
            # ตัวกลางต้องเป็น I_WORD
            for i in range(1, len(seq) - 1):
                if seq[i]['tag'] != 'I_WORD':
                    seq[i]['tag'] = 'I_WORD'
                    corrections += 1
        
        # Rule 4: ตรวจสอบ pattern พยัญชนะ + สระ
        if len(seq) >= 2:
            for i in range(len(seq) - 1):
                curr_char = seq[i]['char']
                next_char = seq[i+1]['char']
                
                # พยัญชนะ + วรรณยุกต์ → มักอยู่ในคำเดียวกัน
                if curr_char in THAI_CONSONANTS and next_char in THAI_TONES:
                    if seq[i]['tag'] == 'E_WORD' and seq[i+1]['tag'] == 'B_WORD':
                        # น่าจะผิด แก้เป็นคำเดียวกัน
                        seq[i]['tag'] = 'I_WORD' if i > 0 else 'B_WORD'
                        seq[i+1]['tag'] = 'E_WORD'
                        corrections += 1
        
        # Rule 5: ตัวการันต์ (์) ต้องอยู่ท้ายคำ
        for i, s in enumerate(seq):
            if s['char'] == THAI_THANTHAKHAT:
                # ตัวการันต์ควรอยู่ท้ายคำ
                if i < len(seq) - 1:
                    # ย้าย tag ให้เหมาะสม
                    for j in range(i+1, len(seq)):
                        seq[j]['tag'] = 'E_WORD'
                    seq[i]['tag'] = 'I_WORD' if i > 0 else 'B_WORD'
                    corrections += 1
    
    print(f"   Corrections made: {corrections}")
    
    # Convert back to predictions dict
    final_predictions = {}
    for seq in sequences:
        for item in seq:
            final_predictions[item['id']] = item['tag']
    
    return final_predictions

# Apply post-processing
final_predictions = apply_thai_post_processing(predictions, test_data)

print(f"\n✅ Post-processed predictions: {len(final_predictions)}")
print(f"   Tag distribution: {dict(Counter(final_predictions.values()))}")


🔧 Applying Thai Post-Processing Rules...
   Found 2067 word sequences
   Corrections made: 10930

✅ Post-processed predictions: 35182
   Tag distribution: {'B_WORD': 1793, 'I_WORD': 29262, 'E_WORD': 4127}


In [39]:
# ถ้ามีเวลา ให้อ่านโมเดลหลายครั้งด้วย seed ต่างกัน

import torch
import numpy as np

def run_multiple_seeds(seeds=[42, 123, 456]):
    """รันโมเดลหลาย seed แล้ว vote"""
    all_predictions = []
    
    for seed in seeds:
        print(f"\n🎲 Running seed {seed}...")
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        # โหลด/ฝึกโมเดลของคุณที่นี่
        # ... (โค้ด training เดิม)
        
        # เก็บ predictions
        # all_predictions.append(seed_predictions)
        pass
    
    # Vote
    if len(all_predictions) > 1:
        final_preds = {}
        for char_id in all_predictions[0].keys():
            votes = [p[char_id] for p in all_predictions]
            final_preds[char_id] = Counter(votes).most_common(1)[0][0]
        return final_preds
    
    return all_predictions[0] if all_predictions else None

# ใช้งาน (ถ้ามีเวลา)
# final_predictions = run_multiple_seeds()

In [40]:
def save_submission(predictions, filepath):
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Id', 'Predicted'])
        for char_id in sorted(predictions.keys()):
            writer.writerow([char_id, predictions[char_id]])
    
    print(f"\n✅ Saved: {filepath}")
    
    df = pd.read_csv(filepath)
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Tags: {set(df['Predicted'].unique())}")
    
    return df

# Save
submission_df = save_submission(final_predictions, 'submission_improved.csv')

# Verify
print("\n🔍 FINAL VERIFICATION")
print("=" * 60)

print(f"✅ Rows: {len(submission_df)} (expected: {len(sample_df)})")
print(f"✅ Match: {'YES' if len(submission_df) == len(sample_df) else 'NO'}")
print(f"✅ Null Values: {submission_df.isnull().sum().sum()}")

if len(submission_df) == len(sample_df) and submission_df.isnull().sum().sum() == 0:
    print("\n🎯 READY TO SUBMIT!")
else:
    print("\n⚠️ FIX ISSUES!")

print("=" * 60)


✅ Saved: submission_improved.csv
   Rows: 35182
   Columns: ['Id', 'Predicted']
   Tags: {'I_WORD', 'B_WORD', 'E_WORD'}

🔍 FINAL VERIFICATION
✅ Rows: 35182 (expected: 35182)
✅ Match: YES
✅ Null Values: 0

🎯 READY TO SUBMIT!
